In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *
from view import *
from reprojection import *


In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,4,9),
#                     dates < datetime(2026,5,1)], axis=0))[0] # CR2310

#t = np.where(np.all([dates > datetime(2026,5,1),
#                     dates < datetime(2026,5,28)], axis=0))[0] # CR2311

#t = np.where(np.all([dates > datetime(2026,5,28),
#                     dates < datetime(2026,6,24)], axis=0))[0] # CR2312

#t = np.where(np.all([dates > datetime(2026,6,24),
#                     dates < datetime(2026,7,22)], axis=0))[0] # CR2313

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,31)], axis=0))[0] # CR2314

t = np.where(np.all([dates > datetime(2026,8,31),
                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,8),
#                     dates < datetime(2026,8,8)], axis=0))[0]

#t = np.where(np.all([dates > datetime(2026,9,14),
#                     dates < datetime(2026,10,13)], axis=0))[0]

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260831T113003_V202610191358C_0648310501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260928T093003_V202610191523C_0649280504.fits'),
 83)

In [3]:
N = 0

LL = 0
GG = 0
LG = 0

LF = 0
GF = 0

FF = 0


for file in files[:]:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    header['CROTA'] = header['CROTA'] + 0.25

    x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1

    data, view = reproject(data, header, correct_dr=True, kind='synoptic', full_output=True)
    data, x, y, alpha = data
    data /= alpha

    x, y = (x + x0 - 1024) / 1024, (y + y0 - 1024) / 1024

    z = 1 / alpha
    n = (~np.isnan(data)).astype(float)


    l = np.expand_dims(z, (-2,-1))
    #g = np.moveaxis(np.array([[x ** 2, x * y, y ** 2, x, y, n]]), (0,1), (-2,-1))
    g = np.moveaxis(np.array([[x, y, n]]), (0,1), (-2,-1))
    f = np.expand_dims(data, (-2,-1))

    ll = np.swapaxes(l, -2, -1) @ l
    gg = np.swapaxes(g, -2, -1) @ g
    lg = np.swapaxes(l, -2, -1) @ g

    lf = np.swapaxes(l, -2, -1) @ f
    gf = np.swapaxes(g, -2, -1) @ f
    ff = np.swapaxes(f, -2, -1) @ f


    N += np.nan_to_num(n)
    w = np.expand_dims(n / N, (-2, -1))

    LL += np.nan_to_num((ll - LL) * w)
    GG += np.nan_to_num((gg - GG) * w)
    LG += np.nan_to_num((lg - LG) * w)
    LF += np.nan_to_num((lf - LF) * w)
    GF += np.nan_to_num((gf - GF) * w)
    FF += np.nan_to_num((ff - FF) * w)


LL[N == 0] = np.nan
GG[N == 0] = np.nan
LG[N == 0] = np.nan

LF[N == 0] = np.nan
GF[N == 0] = np.nan
FF[N == 0] = np.nan

N[N == 0] = np.nan

/tmp/ipykernel_290896/2162783307.py:54: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))


In [4]:
LLi = np.linalg.inv(LL)

Q = LLi @ LF
R = LLi @ LG
GL = np.swapaxes(LG, -2, -1)


S = (FF - GF[:,:,-1:,:] ** 2 - Q ** 2 * (LL - GL[:,:,-1:,:] ** 2)) / LL / np.expand_dims(N - 1, (-1,-2))

D = np.linalg.inv(np.nanmean((GG - GL @ R) / S, axis=(0,1)))
V = np.nanmean((GF - GL @ Q) / S, axis=(0,1))
P = D @ V
dQ = R @ P

Q_ = Q - dQ

S = np.squeeze(S)
P = np.squeeze(P)
Q = np.squeeze(Q)
Q_ = np.squeeze(Q_)
dQ = np.squeeze(dQ)

print(P)

[-0.42663842  0.0749057  -0.23250607]


In [5]:
show_map(np.sqrt(S), view, vmin=0, vmax=5, cmap='inferno')

In [6]:
show_map(Q_, view, vmin=-500, vmax=500)

In [7]:
show_map(dQ, view, vmin=-5, vmax=5, cmap='seismic')

In [8]:
xi, yi = np.mgrid[:2048,:2048]
xi, yi = xi / 1024 - 1, yi / 1024 - 1

#bias = P[0] * xi ** 2 + P[1] * xi * yi + P[2] * yi ** 2 + P[3] * xi + P[4] * yi + P[5]
bias = P[0] * xi + P[1] * yi + P[2]

In [6]:
nx = len(Q)

lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

flux = np.nanmean(Q, axis=1)
flux_ = np.nanmean(Q_, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux)
plt.plot(lat, flux_)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_293079/3062491754.py:5: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(Q, axis=1)


In [7]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2315.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.09.12_09:28:02_TAI'

In [8]:
show_map(data_, view, vmin=-500, vmax=500)

In [9]:
#flux_hmi = np.nanmean(data_ * Z2, axis=1) / np.nanmean(Z2, axis=1)
flux_hmi = np.nanmean(data_, axis=1)

nx, ny = data_.shape
lat_hmi = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_hmi, flux_hmi, label='HMI')
plt.plot(lat, flux, label='FDT')
#plt.plot(lat, flux_, label='FDT (corrected)', color='red')

plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_293079/3123798801.py:2: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_, axis=1)
